In [1]:
### 相对于22.2 版本 当前版用于多品种

In [2]:
import pandas  as pd
import numpy as np
import pdb, os, datetime, itertools, time, hashlib
import pyprojroot, sys
from pyprojroot.criterion import has_file
sys.path.insert(0, str(pyprojroot.find_root(has_file("pyproject.toml"))))
from dotenv import load_dotenv

load_dotenv()
from lib.flp001 import *

/workspace/worker/pj/Chrono/genuis/mizar
/workspace/worker/pj/Chrono/genuis/mizar/config/contract.toml


In [3]:
method = 'ricso2'
task_id = '119001'  # 公式来源品种的 task_id，与比较脚本的顶层参数一致
session = '20260605'  # 填写比较脚本实际使用的评估批次
period = 15
category = 1
instruments = 'ifs'  # 公式来源品种，也是判断 direction 所用的品种
compare_instruments = ['ifs', 'ics', 'ihs']  # 按实际参与比较的品种填写


In [4]:
results = fetch_data(method=method, task_id=task_id, instruments=instruments,
                     period=period, session=session, category=category, is_multi=True)
results['name'] = results['name'].astype(int).astype(str)
for column in ['ic_mean', 'total_ic', 'ann_sharpe', 'calmar']:
    results[column] = pd.to_numeric(results[column], errors='coerce')
results['abs_ic_mean'] = results['ic_mean'].abs()
results['abs_total_ic'] = results['total_ic'].abs()

/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/ifs/rulex/119001/nxt1_ret_15h/20260605


In [5]:
# 将原来的 left_results / right_results 扩展为按品种保存的字典
asset_results = {
    symbol: results[results['instrument'] == symbol].copy()
    for symbol in compare_instruments
}

view_cols = ['instrument', 'name', 'expression', 'avg_ret', 'total_ret',
             'max_dd', 'calmar', 'win_rate', 'pl_ratio', 'ic_mean', 'icir',
             'sharpe', 'ann_sharpe', 'total_ic']
for symbol in compare_instruments:
    print(symbol)
    display(asset_results[symbol][view_cols].head())


ifs


,instrument,name,expression,avg_ret,total_ret,max_dd,calmar,win_rate,pl_ratio,ic_mean,icir,sharpe,ann_sharpe,total_ic
0,ifs,10015126,"MMinDiff(90,MIR(10,'tc009_1_1_2_3_0'))",0.48,879.79,-10.08,1.85,50.82,1.08,-0.0282,-0.1030,0.03,1.80,-0.0301
4,ifs,10045496,"MDPO(240,EMA(60,'oi039_2_3_1'))",0.46,794.97,-12.88,1.39,50.96,1.06,0.0309,0.1154,0.03,1.62,0.0264
8,ifs,10075137,"MADiff(15,MADecay(120,'tc004_2_2_3_0'))",0.47,857.56,-12.53,1.47,50.98,1.07,-0.0320,-0.1167,0.03,1.77,-0.0291
12,ifs,10125007,"EMA(10,MRANK(60,'tc005_2_2_3_1'))",0.41,624.34,-9.44,1.70,50.88,1.06,-0.0321,-0.1158,0.03,1.60,-0.0263
16,ifs,10155371,"MDEMA(15,MPERCENT(5,'tc018_2_3_1'))",0.39,556.25,-12.37,1.23,50.56,1.07,-0.0213,-0.0754,0.02,1.50,-0.0247


ics


,instrument,name,expression,avg_ret,total_ret,max_dd,calmar,win_rate,pl_ratio,ic_mean,icir,sharpe,ann_sharpe,total_ic
1,ics,10015126,"MMinDiff(90,MIR(10,'tc009_1_1_2_3_0'))",0.50,451.39,-19.60,0.95,51.02,1.06,-0.0283,-0.1025,0.03,1.55,-0.0246
5,ics,10045496,"MDPO(240,EMA(60,'oi039_2_3_1'))",0.55,552.65,-23.75,0.86,51.36,1.04,0.0292,0.1098,0.02,1.48,0.0248
9,ics,10075137,"MADiff(15,MADecay(120,'tc004_2_2_3_0'))",0.50,460.19,-26.48,0.71,51.39,1.04,-0.0279,-0.1012,0.02,1.45,-0.0244
13,ics,10125007,"EMA(10,MRANK(60,'tc005_2_2_3_1'))",0.55,570.44,-17.54,1.19,51.17,1.06,-0.0313,-0.1130,0.03,1.62,-0.0273
17,ics,10155371,"MDEMA(15,MPERCENT(5,'tc018_2_3_1'))",0.50,460.70,-16.16,1.16,50.84,1.06,-0.0217,-0.0768,0.02,1.46,-0.0246


ihs


,instrument,name,expression,avg_ret,total_ret,max_dd,calmar,win_rate,pl_ratio,ic_mean,icir,sharpe,ann_sharpe,total_ic
2,ihs,10015126,"MMinDiff(90,MIR(10,'tc009_1_1_2_3_0'))",0.30,171.06,-14.76,0.71,50.78,1.04,-0.0181,-0.0660,0.02,1.06,-0.0186
6,ihs,10045496,"MDPO(240,EMA(60,'oi039_2_3_1'))",0.31,180.10,-16.63,0.65,50.97,1.03,0.0184,0.0679,0.02,1.04,0.0175
10,ihs,10075137,"MADiff(15,MADecay(120,'tc004_2_2_3_0'))",0.37,262.34,-16.64,0.82,50.84,1.05,-0.0231,-0.0842,0.02,1.38,-0.0231
14,ihs,10125007,"EMA(10,MRANK(60,'tc005_2_2_3_1'))",0.39,284.66,-14.31,1.00,51.08,1.05,-0.0275,-0.1001,0.03,1.50,-0.0249
18,ihs,10155371,"MDEMA(15,MPERCENT(5,'tc018_2_3_1'))",0.33,219.50,-16.81,0.73,50.72,1.05,-0.0142,-0.0512,0.02,1.26,-0.0212


### 初步筛选：所有指定品种均达到阈值

In [6]:
ic_mean = 0.02
total_ic = 0.012
ann_sharpe = 1
calmar = 1

In [7]:
# 与原 notebook 一样，分别筛选每个品种，再取交集
filtered_results = {}
for symbol in compare_instruments:
    data = asset_results[symbol]
    filtered_results[symbol] = data[
        (data['abs_ic_mean'] >= ic_mean)
        & (data['abs_total_ic'] >= total_ic)
        & (data['ann_sharpe'] >= ann_sharpe)
        & (data['calmar'] >= calmar)
    ].copy()

pd.DataFrame([
    {'instrument': symbol, 'total': len(asset_results[symbol]),
     'passed': len(filtered_results[symbol])}
    for symbol in compare_instruments
])


,instrument,total,passed
0,ifs,60,60
1,ics,60,29
2,ihs,60,1


In [8]:
# 关注的核心绩效指标，与原 notebook 一致
perf_cols = ['ann_sharpe', 'calmar', 'abs_ic_mean', 'abs_total_ic',
             'max_dd', 'avg_ret', 'total_ic', 'ic_mean']
key_cols = ['name', 'expression']

# 多次 inner merge：缺少任一品种，或任一品种未达标，均不会保留
merged_results = None
for symbol in compare_instruments:
    data = filtered_results[symbol][key_cols + perf_cols].rename(
        columns={column: f'{column}_{symbol}' for column in perf_cols}
    )
    if merged_results is None:
        merged_results = data.copy()
    else:
        merged_results = pd.merge(merged_results, data, on=key_cols,
                                  how='inner', validate='one_to_one')

# 同一指标的各品种放在一起，方便直接比较
# 重排后再显示，原始 IC 的正负方向保持不变
display_cols = key_cols + [f'{column}_{symbol}'
                           for column in perf_cols for symbol in compare_instruments]
merged_results = merged_results[display_cols]
display(merged_results.sort_values(by=[f'ann_sharpe_{instruments}']))


,name,expression,ann_sharpe_ifs,ann_sharpe_ics,ann_sharpe_ihs,calmar_ifs,calmar_ics,calmar_ihs,abs_ic_mean_ifs,abs_ic_mean_ics,...,max_dd_ihs,avg_ret_ifs,avg_ret_ics,avg_ret_ihs,total_ic_ifs,total_ic_ics,total_ic_ihs,ic_mean_ifs,ic_mean_ics,ic_mean_ihs
0,10125007,"EMA(10,MRANK(60,'tc005_2_2_3_1'))",1.6,1.62,1.5,1.7,1.19,1.0,0.0321,0.0313,...,-14.31,0.41,0.55,0.39,-0.0263,-0.0273,-0.0249,-0.0321,-0.0313,-0.0275


In [9]:
# 与原 notebook 一样，仅用公式来源品种判断方向
# 四个阈值使用绝对 IC；这里仍使用原始、有正负号的 IC
total_ic_col = f'total_ic_{instruments}'
ic_mean_col = f'ic_mean_{instruments}'
conditions = [
    (merged_results[total_ic_col] > 0) & (merged_results[ic_mean_col] > 0),
    (merged_results[total_ic_col] < 0) & (merged_results[ic_mean_col] < 0)
]
merged_results['direction'] = np.select(conditions, [1, -1], default=0)

# 图表路径仍指向公式来源品种下的同一批次
session_name = f'd{session}' if category == 2 else str(session)
session_path = (Path(base_path) / method / instruments / 'rulex' / str(task_id)
                / f'nxt1_ret_{period}h' / session_name)
merged_results['plot'] = merged_results['name'].apply(
    lambda name: str(session_path / 'plot' / f'{name}.png')
)

merged_results = merged_results[['expression', 'plot', 'name', 'direction']].rename(
    columns={'name': 'factor_id', 'expression': 'formula'}
)
merged_results['plot'] = merged_results['plot'].apply(make_clickable)


In [10]:
to_html(merged_results)


url,formula,factor_id,direction,plot
10125007,"EMA(10,MRANK(60,'tc005_2_2_3_1'))",10125007,-1,/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/ifs/rulex/119001/nxt1_ret_15h/20260605/plot/10125007.png


In [11]:
# 转成原 notebook 的临时 cohort 格式
merged_results1 = merged_results[['formula', 'direction']].copy()
merged_results1['source'] = session
merged_results1['category'] = 'p'
merged_results1.head()


,formula,direction,source,category
0,"EMA(10,MRANK(60,'tc005_2_2_3_1'))",-1,20260605,p


In [12]:
file_path = (Path(base_path) / method / instruments / 'rulex' / str(task_id)
             / f'nxt1_ret_{period}h')
file_path


PosixPath('/workspace/worker/pj/Chrono/genuis/mizar/records/ricso2/ifs/rulex/119001/nxt1_ret_15h')

In [13]:
merged_results1.to_csv(file_path / f'{session}_cohort.csv',
                       encoding='UTF-8', index=False)
